In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

    def fit(self, X, y):
        n_samples, _ = X.shape
        w = np.full(n_samples, 1 / n_samples)

        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)

            err = np.sum(w * (y_pred != y)) / np.sum(w)
            alpha = 0.5 * np.log((1 - err) / (err + 1e-10))

            w *= np.exp(-alpha * y * y_pred)
            w /= np.sum(w)  # Normalize

            self.models.append(stump)
            self.alphas.append(alpha)

    def predict(self, X):
        clf_preds = [alpha * clf.predict(X) for clf, alpha in zip(self.models, self.alphas)]
        return np.sign(np.sum(clf_preds, axis=0))


In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Create binary classification data with labels as -1 and 1
X, y = make_classification(n_samples=100, n_features=2, n_informative=2, n_redundant=0)
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train AdaBoost
model = AdaBoost(n_estimators=10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Accuracy
accuracy = np.mean(y_pred == y_test)
print("Accuracy:", accuracy)


Accuracy: 0.9
